# PSF Extraction Tester

Simple notebook to:
1. Load a Rubin deepCoadd via Butler
2. Extract the PSF at the image center
3. Extract PSFs at many positions across the image
4. Check FWHM, shape, and spatial variation

**Run this on the RSP** (data.lsst.cloud).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# Debug: show where we are
print(f'Working dir : {os.getcwd()}')
print(f'User        : {os.environ.get("USER", "unknown")})
print(f'Home        : {os.path.expanduser("~")})
print(f'Hostname    : {os.environ.get("HOSTNAME", "unknown")})
print()

## 1. Connect to Butler and load coadd

In [ ]:
from lsst.daf.butler import Butler
from lsst.geom import Point2D

print('✓ LSST imports OK')

In [ ]:
DATA_ID = {'tract': 4431, 'patch': 17, 'band': 'i'}

# All known DP0.2 Butler configs for RSP
butler_configs = [
    # Current RSP (2024+)
    ('dp02',                            '2.2i/runs/DP0.2'),
    # S3 bucket
    ('s3://butler-us-central1-dp02',    '2.2i/runs/DP0.2'),
    # Direct repo path
    ('/repo/dc2',                       '2.2i/runs/DP0.2'),
    # USDF
    ('/sdf/group/rubin/repo/dc2',       '2.2i/runs/DP0.2'),
    ('/repo/main',                      '2.2i/runs/DP0.2'),
]

butler = None
exposure = None

for repo, collection in butler_configs:
    try:
        print(f'Trying: repo={repo!r}, collection={collection!r}')
        b = Butler(repo, collections=collection)
        print(f'  Butler created, now loading deepCoadd...')
        exp = b.get('deepCoadd', dataId=DATA_ID)
        butler = b
        exposure = exp
        print(f'  ✓ Success!')
        break
    except Exception as e:
        print(f'  ✗ {type(e).__name__}: {str(e)[:100]}')

if exposure is None:
    raise RuntimeError(
        'Could not load coadd from any Butler config.\n'
        'Are you on the RSP? Check: https://data.lsst.cloud\n'
        'Try running in a terminal: python -c "from lsst.daf.butler import Butler; print(Butler.__file__)"'
    )

image = exposure.image.array
print(f'\nImage shape : {image.shape}')
print(f'Image dtype : {image.dtype}')
print(f'Image range : [{image.min():.2f}, {image.max():.2f}]')

## 2. Check PSF model

In [ ]:
psf_model = exposure.getPsf()
print(f'PSF model type : {type(psf_model).__name__}')
print(f'PSF model      : {psf_model}')
assert psf_model is not None, 'No PSF attached to this exposure!'
print('\n✓ PSF model exists')

## 3. Extract PSF at image center

In [ ]:
cy, cx = image.shape[0] // 2, image.shape[1] // 2
center = Point2D(float(cx), float(cy))

psf_im = psf_model.computeImage(center)
psf_arr = psf_im.array.copy()

shape = psf_model.computeShape(center)
ixx, iyy, ixy = shape.getIxx(), shape.getIyy(), shape.getIxy()
sigma = np.sqrt((ixx + iyy) / 2.0)
fwhm = 2.355 * sigma

print(f'Position       : ({cx}, {cy})')
print(f'Kernel shape   : {psf_arr.shape}')
print(f'Kernel sum     : {psf_arr.sum():.6f}')
print(f'Kernel max     : {psf_arr.max():.6f}')
print(f'Ixx            : {ixx:.4f}')
print(f'Iyy            : {iyy:.4f}')
print(f'Ixy            : {ixy:.4f}')
print(f'sigma          : {sigma:.4f} px')
print(f'FWHM           : {fwhm:.4f} px  =  {fwhm * 0.2:.4f} arcsec')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(psf_arr, cmap='inferno', origin='lower')
axes[0].set_title(f'PSF at center  ({psf_arr.shape})')

psf_clip = np.clip(psf_arr, 1e-10, None)
axes[1].imshow(psf_clip, cmap='inferno', origin='lower',
               norm=LogNorm(vmin=psf_clip[psf_clip > 0].min(), vmax=psf_clip.max()))
axes[1].set_title('Log scale (wings)')

plt.tight_layout()
plt.show()

## 4. Extract PSFs across the image

Sample a grid of positions and check whether `computeImage` and `computeShape` succeed everywhere.

In [ ]:
ny, nx = image.shape
margin = 200
n_side = 5  # 5×5 = 25 sample points

xs = np.linspace(margin, nx - margin, n_side)
ys = np.linspace(margin, ny - margin, n_side)

results = []
failures = []

for y in ys:
    for x in xs:
        pos = Point2D(float(x), float(y))
        try:
            arr = psf_model.computeImage(pos).array.copy()
            sh = psf_model.computeShape(pos)
            s = np.sqrt((sh.getIxx() + sh.getIyy()) / 2.0)
            fw = 2.355 * s
            results.append({'x': x, 'y': y, 'fwhm': fw,
                            'ixx': sh.getIxx(), 'iyy': sh.getIyy(),
                            'ixy': sh.getIxy(),
                            'kernel_shape': arr.shape,
                            'kernel_sum': arr.sum(),
                            'psf': arr})
        except Exception as e:
            failures.append({'x': x, 'y': y, 'error': str(e)})

print(f'Succeeded: {len(results)} / {n_side**2}')
print(f'Failed   : {len(failures)}')
if failures:
    for f in failures:
        print(f"  FAIL at ({f['x']:.0f}, {f['y']:.0f}): {f['error'][:60]}")

In [ ]:
# Print table
header = f'{"X":>8} {"Y":>8} {"FWHM(px)":>10} {"FWHM(as)":>10} {"Ixx":>8} {"Iyy":>8} {"Ixy":>8} {"sum":>8} {"shape":>10}'
print(header)
print('-' * len(header))
for r in results:
    print(f"{r['x']:8.0f} {r['y']:8.0f} {r['fwhm']:10.4f} {r['fwhm']*0.2:10.4f} "
          f"{r['ixx']:8.4f} {r['iyy']:8.4f} {r['ixy']:8.4f} {r['kernel_sum']:8.4f} {str(r['kernel_shape']):>10}")

In [ ]:
# FWHM variation
fwhms = [r['fwhm'] for r in results]
print(f'FWHM min  : {min(fwhms):.4f} px  ({min(fwhms)*0.2:.4f} arcsec)')
print(f'FWHM max  : {max(fwhms):.4f} px  ({max(fwhms)*0.2:.4f} arcsec)')
print(f'FWHM mean : {np.mean(fwhms):.4f} px')
print(f'FWHM std  : {np.std(fwhms):.4f} px')
print(f'Variation : {max(fwhms)-min(fwhms)::.4f} px  ({(max(fwhms)-min(fwhms))*0.2:.4f} arcsec)')

if np.std(fwhms) < 0.001:
    print('\n→ FWHM is constant (expected for mock PSF or single-epoch coadd)')
else:
    print(f'\n→ PSF varies spatially by {(max(fwhms)-min(fwhms))/np.mean(fwhms)*100:.1f}%')

## 5. Visualize PSFs across the image

In [ ]:
fig, axes = plt.subplots(n_side, n_side, figsize=(2.5 * n_side, 2.5 * n_side))

for idx, ax in enumerate(axes.flat):
    if idx < len(results):
        r = results[idx]
        ax.imshow(r['psf'], cmap='inferno', origin='lower')
        ax.set_title(f"({r['x']:.0f},{r['y']:.0f})\nFWHM={r['fwhm']:.2f}px", fontsize=8)
    else:
        ax.axis('off')
    ax.tick_params(labelsize=5)

plt.suptitle(f'PSF grid ({n_side}x{n_side}) — {DATA_ID}', fontsize=14)
plt.tight_layout()
plt.show()

## 6. FWHM map

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

vmin_img, vmax_img = np.percentile(image, [5, 95])
ax.imshow(image, cmap='gray', origin='lower', vmin=vmin_img, vmax=vmax_img, alpha=0.5)

xs_plot = [r['x'] for r in results]
ys_plot = [r['y'] for r in results]
fwhms_plot = [r['fwhm'] for r in results]

sc = ax.scatter(xs_plot, ys_plot, c=fwhms_plot, cmap='coolwarm', s=120,
                edgecolors='black', linewidths=0.5, zorder=5)
plt.colorbar(sc, ax=ax, label='FWHM (pixels)', shrink=0.8)

for r in results:
    ax.annotate(f"{r['fwhm']:.2f}", (r['x'], r['y']),
                textcoords='offset points', xytext=(5, 5), fontsize=7, color='white')

for f in failures:
    ax.plot(f['x'], f['y'], 'rx', ms=12, mew=2)

ax.set_title(f'PSF FWHM across coadd — {DATA_ID}\n'
             f'range: [{min(fwhms):.3f}, {max(fwhms):.3f}] px'
             f'{"  [MOCK]" if USE_MOCK else ""}', fontsize=12)
ax.set_xlabel('X (pixels)')
ax.set_ylabel('Y (pixels)')
plt.tight_layout()
plt.show()

## 7. Radial profiles at a few positions

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

indices = [0, n_side - 1, len(results) // 2, -n_side, -1]
indices = [i % len(results) for i in indices]
indices = list(dict.fromkeys(indices))  # deduplicate, preserve order

for idx in indices:
    r = results[idx]
    psf = r['psf']
    cy_k, cx_k = psf.shape[0] // 2, psf.shape[1] // 2
    yy, xx = np.mgrid[:psf.shape[0], :psf.shape[1]]
    rr = np.sqrt((xx - cx_k)**2 + (yy - cy_k)**2).ravel()
    vals = psf.ravel()
    order = np.argsort(rr)
    ax.plot(rr[order], vals[order], '.', ms=1, alpha=0.4,
            label=f"({r['x']:.0f},{r['y']:.0f}) FWHM={r['fwhm']:.2f}")

ax.set_xlabel('Radius (pixels)')
ax.set_ylabel('PSF value')
ax.set_yscale('log')
ax.set_ylim(bottom=1e-6)
ax.set_title('PSF radial profiles at selected positions')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Test edge cases

Try positions near the image edges and corners to see where the PSF model breaks.

In [ ]:
edge_positions = [
    (0, 0), (0, nx//2), (0, nx-1),
    (ny//2, 0), (ny//2, nx-1),
    (ny-1, 0), (ny-1, nx//2), (ny-1, nx-1),
    (50, 50), (ny-50, nx-50),
]

print(f'{"Y":>6} {"X":>6} {"Status":>10} {"FWHM":>8} {"Details"}')
print('-' * 60)

n_ok, n_fail = 0, 0
for (ey, ex) in edge_positions:
    pos = Point2D(float(ex), float(ey))
    try:
        arr = psf_model.computeImage(pos).array
        sh = psf_model.computeShape(pos)
        s = np.sqrt((sh.getIxx() + sh.getIyy()) / 2.0)
        fw = 2.355 * s
        print(f'{ey:6d} {ex:6d} {"OK":>10} {fw:8.3f} shape={arr.shape}, sum={arr.sum():.4f}')
        n_ok += 1
    except Exception as e:
        print(f'{ey:6d} {ex:6d} {"FAILED":>10} {"---":>8} {str(e)[:40]}')
        n_fail += 1

print(f'\nEdge test: {n_ok} OK, {n_fail} failed')
if n_fail > 0:
    print('→ Use edge_buffer >= 100 px when placing injected clusters')

## Summary

In [ ]:
print('=' * 50)
print('PSF EXTRACTION TEST RESULTS')
print('=' * 50)
print(f'Data source    : {"Real coadd" if not USE_MOCK else "MOCK (run on RSP for real data)"}')
print(f'Image shape    : {image.shape}')
print(f'PSF model type : {type(psf_model).__name__}')
print(f'Grid test      : {len(results)}/{n_side**2} succeeded, {len(failures)} failed')
print(f'FWHM mean      : {np.mean(fwhms):.4f} px = {np.mean(fwhms)*0.2:.4f} arcsec')
print(f'FWHM range     : [{min(fwhms):.4f}, {max(fwhms):.4f}] px')
print(f'FWHM std       : {np.std(fwhms):.4f} px')
print()
print('Checklist:')
print(f'  [{"✓" if not USE_MOCK else "✗"}] Butler loaded real coadd')
print(f'  [✓] PSF model exists')
print(f'  [{"✓" if len(results) > 0 else "✗"}] computeImage works')
print(f'  [{"✓" if len(results) > 0 else "✗"}] computeShape works')
print(f'  [{"✓" if len(failures) == 0 else "~"}] All grid points succeeded')
if USE_MOCK:
    print()
    print('⚠️  Re-run on the RSP to test with real Rubin PSFs.')
    print('   Log in at: https://data.lsst.cloud')